# L1c: A First Tested Engineering Calculation

We will translate the ideal-gas relation, its unit contract, and its admissible input domain into a small Julia function with executable tests.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __State the contract before writing code:__ Fix the units, the admissible input domain, and the return value of an engineering calculation before translating it into a Julia function. The contract is what makes the function testable.
> * __Reject inputs the model cannot support:__ Raise a clear error when an argument violates a physical assumption, rather than returning a number that looks plausible and means nothing.
> * __Test known behavior, not just execution:__ Check a reference case with an appropriate numerical tolerance and interpret the result in engineering units, so that a passing test says something about the calculation.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

___

## Contract before code

We will accept amount in mol, temperature in K, and volume in $\mathrm{m^3}$. We will return pressure in Pa using $R=8.31446261815324\;\mathrm{Pa\,m^3/(mol\,K)}$. Amount, absolute temperature, volume, and the gas constant must be finite and positive.

___

## Inspect and Edit the Function

The executable implementation lives in `src/Compute.jl`, where it can be tested independently of notebook state. Read the contract and validation loop, then edit that source file during class as we build the function.

In [ ]:
implementation_path = joinpath(CHEME5800_L1C_ROOT, "src", "Compute.jl")
implementation_preview = read(implementation_path, String)
println(implementation_preview) # println, not the bare value: we want readable lines, not one escaped string

___

## Compute a reference case

One mole of an ideal gas near standard temperature in approximately $22.414$ L should have a pressure near one atmosphere.

In [ ]:
amount_mol = 1.0
temperature_K = 273.15
volume_m3 = 0.02241396954
pressure_Pa = ideal_gas_pressure(amount_mol, temperature_K, volume_m3)
pressure_kPa = pressure_Pa / 1000
(pressure_Pa = pressure_Pa, pressure_kPa = pressure_kPa)

___

## Test the contract

Floating-point results should normally be compared with `isapprox`, not exact equality. Invalid physical inputs should fail clearly.

In [ ]:

@testset "ideal-gas pressure contract" begin
    @test isapprox(pressure_Pa, 101_325.0; rtol = 1e-8)
    @test ideal_gas_pressure(2, 300, 0.05) isa Float64
    @test_throws ArgumentError ideal_gas_pressure(0, 300, 0.05)
    @test_throws ArgumentError ideal_gas_pressure(1, -10, 0.05)
    @test_throws ArgumentError ideal_gas_pressure(1, 300, Inf)
end

___

## Change, predict, compute, explain

Before running the next cell, predict what happens when temperature doubles while amount and volume remain fixed.

In [ ]:
baseline_pressure = ideal_gas_pressure(1.0, 300.0, 0.025)
hot_pressure = ideal_gas_pressure(1.0, 600.0, 0.025)
pressure_ratio = hot_pressure / baseline_pressure
@test pressure_ratio == 2.0
pressure_ratio

> __Wait — didn't we just say not to use `==`?__ We did, and this is the exception worth understanding. Doubling is _exact_ in binary floating point: $600.0 = 2\times 300.0$ with no rounding, and scaling by a power of two only increments the exponent field, leaving every significand bit untouched. So the two pressures have identical significands and the ratio is exactly `2.0`.
>
> This is fragile in a way that is easy to miss. Change `600.0` to `900.0` and predict the result before you run it: the ratio comes back `2.9999999999999996`, and `== 3.0` fails. Tripling is not a power of two, so the rounding no longer cancels. Exact equality is safe only when you can point to the reason — otherwise, reach for `isapprox`.

___

## Interpretation

One mole in 22.4 L at 273.15 K came back as roughly $101$ kPa — about one atmosphere, which is the number to carry in your head as a sanity check for gas-phase work. The implementation reproduces that reference pressure and the expected proportional response to temperature. It does **not** establish that a real gas is ideal under every condition. The model assumption remains part of the result.

___

## Summary
Translating an equation into code is the easy half; stating what the code promises, and testing that promise, is the half that makes the result usable.

> __Key Takeaways:__
>
> * **Units and domain come first:** Deciding what the arguments mean and which values are admissible turns an equation into a function whose behavior can be checked.
> * **Invalid input deserves an error:** Rejecting a non-physical argument with an `ArgumentError` is more useful than returning a finite number that carries no meaning.
> * **A passing test validates the implementation, not the model:** These tests show the function matches its contract. They say nothing about whether a real gas behaves ideally under the conditions you care about.

Every calculation in this course carries assumptions. Writing them down as a contract is what lets a test tell you when they have been violated.
___